# Proposal: quantum encodings and reservoir on Track A

## Goal

Extend the quantum feature tracks built on Track B (static encodings `Q_{encoding}_{getter}` from `quantum_data_prep.ipynb`, multi-circuit reservoir `MCR_su_z` from `quantum_reservoir_prep.ipynb`) to the full stationary panel, Track A (123 features × 767 months). The question is whether quantum features still help when they get the full panel instead of a few hand-picked indicators.

## Current setup (Track B)

- Inputs: `track_b_{target}.parquet` (10–12 features), causal (expanding) min-max scaled to [0, 1], encoded as angles in [0, 2π].
- Static encodings, one circuit per month: `su` (EfficientSU2, `ceil(n/2)` qubits), `heisenberg` (Trotterized XX+YY+ZZ chain, `n+1` qubits, 6 steps), `zz` (ZZ-only chain, `n+1` qubits). Readouts: `pce`, `pauli`, `single_bit`, `bitstring` (only up to 8 qubits).
- Reservoir `MCR_su_z`: 12-month trailing window run as 12 sequential `su` circuits. The ⟨Z⟩ readout is fed back as a hidden state (as many hidden values as inputs), with a fixed random EfficientSU2 ansatz. 10–12 qubits.
- Simulation: Aer statevector, estimator precision 0.01, 4096 shots, no noise. Build time: 17.8 min for all static tracks and 10.1 min for the reservoir (all 4 targets).
- Evaluation: 647 rolling folds (120-month train, 1-month test), targets INDPRO, PAYEMS, CPIAUCSL and S&P 500, horizons 1, 3 and 6, 10 models (trees, linear/signature kernels, ridge readout, RNN/GRU/LSTM) plus random-walk and AR(6) baselines.

## Track B results

Ratios are RMSE on the given track over RMSE on Track B for the same model, target, horizon and folds, averaged over 10 models × 4 targets × 3 horizons (120 cells). Below 1 means the track beats raw Track B.

| Track | Mean RMSE / B | Cells better than B |
|---|---|---|
| MCR_su_z (reservoir) | 0.973 | 58% |
| Q_su_single_bit | 0.990 | 41% |
| Q_zz_pce | 0.995 | 38% |
| Q_zz_single_bit | 0.996 | 41% |
| Q_su_pce | 0.998 | 51% |
| other heisenberg / pauli tracks | 0.998–1.005 | 31–38% |
| Q_su_bitstring | 1.092 | 22% |

- The reservoir is the only quantum track with a consistent gain, and the gain grows with horizon: 0.988 at h=1, 0.975 at h=3, 0.957 at h=6. It helps the linear learners most (ridge readout 0.92, linear SVM 0.92, AdaBoost 0.94).
- Static encodings are at parity with their raw inputs. Bitstring probabilities hurt.
- Lowest mean MASE overall: signature kernel on Track A (0.998), then on `Q_zz_pauli` (1.000) and `Q_heisenberg_pauli` (1.006). All models are close to the in-sample naive forecast (MASE = 1).
- Classical Track A vs B: tree models and the signature kernel do 2–3.5% better on A (random forest 0.965, XGBoost 0.967, signature kernel 0.974). Linear SVM does twice as badly (2.0), which pulls A's average to 1.108. So the wider panel has information that the nonlinear learners can use.
- No significance test has been run yet. Most differences are a few percent.

## Scaling problem

| Pipeline | Track B qubits | Track A qubits |
|---|---|---|
| su (static) | 5–6 | 62 |
| heisenberg / zz (static) | 11–13 | 124 |
| MCR_su_z (123 inputs + 123 hidden) | 10–12 | 123 |
| MCR_su_z with 15 hidden values | — | 69 |

Statevector memory doubles with each qubit and tops out near 30 qubits on a laptop, so none of these pipelines can run on Track A unchanged.

## Proposed approach

Use Aer `matrix_product_state` (MPS) simulation, whose cost depends on entanglement rather than qubit count. Priorities follow the Track B results:

1. `MCR_su_z` on Track A (main deliverable). Switch the `su` encoding from `entanglement="full"` to `"reverse_linear"` (same unitary, n−1 nearest-neighbour CNOTs). Each step is then two CNOT chains plus single-qubit rotations, which should keep the bond dimension small. Run with `NUM_HIDDEN` = 123 and 15 to test whether a smaller hidden state loses information.
2. Static `su` and `zz` with `single_bit` and `pce` readouts. These are cheap under MPS (bond dimension ≤ 2), but low priority given parity on Track B.
3. Dropped: `bitstring` (2^62+ columns, worst on B) and exact `heisenberg` (bond dimension up to 4^6 = 4096). Truncated heisenberg only if time allows, reported as approximate.
4. Reduced-panel comparison: the same pipelines on 15 causally fitted (expanding-window) principal components of Track A. That gives 8–16 qubits for static tracks and 15 for the reservoir, all feasible as statevector. This separates "more information" from "more qubits".

## Evaluation

- Same loop, folds, targets, horizons and models as the existing tracks (`02_evaluation_loop`), with plots and rankings from `03_plot_results` and `04_model_rankings`.
- Primary comparison: each quantum track on A vs classical Track A, as quantum-on-B was compared with B. Secondary: quantum on A vs quantum on B, and full panel vs PCA.
- Add a Diebold–Mariano test per model, target and horizon, since Track B gains are a few percent.
- Validate MPS output against exact statevector on a subset of 20 features or fewer.

## Compute estimate

On Track B, building features is cheap (minutes). The evaluation loop dominates: the classical Track A run took 1h19m for 6 of the 10 models. MPS cost at 62–124 qubits has not been benchmarked. First step: time 25 windows of `MCR_su_z` and 25 rows of each static track before committing to full runs. Expected: tens of minutes to a few hours per Track A quantum track to build, plus one to a few hours each to evaluate.

## Risks

- MPS bond dimension may grow more than expected for the reservoir, because the fed-back hidden state is classical but re-encoded at every step.
- With 123 inputs, expectation values may concentrate near zero (barren, uninformative features), especially for `pce` and `single_bit`.
- Linear SVM already degrades on Track A, so model-specific results may reflect feature count rather than quantum structure. The PCA comparison is there to check this.
- All results are noiseless. They say nothing about hardware performance at this width.